<a href="https://colab.research.google.com/github/lygitdata/GarmentIQ/blob/main/test/tutorial_matting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial - GarmentIQ Matting

Segmentation gives every pixel a yes or no answer, which leaves hard, stair-stepped edges
and discards semi-transparent detail such as lace or loose fibers. **Matting** instead
predicts a continuous alpha value per pixel, so a garment composites onto a new background
without a cut-out look.

Matting always needs guidance from a segmentation step first. This tutorial covers the two
models GarmentIQ supports, ViTMatte and Matting Anything, the trimap that guides them, and
three practical pairings.

## Table of Contents

1. [Prerequisites](#prerequisites)
2. [Matting with ViTMatte and a trimap](#vitmatte)
3. [Matting with Matting Anything](#mam)
4. [Matting from a text prompt with SAM 3](#sam3)
5. [Choosing a pairing](#choosing)

<a name="prerequisites"></a>
## Prerequisites

Install the package and download the test image and model weights. On Colab you can keep
this section collapsed.

In [ ]:
# @title Install GarmentIQ
!pip install garmentiq -q

In [ ]:
# @title Import GarmentIQ and choose a device

import numpy as np
import torch

import garmentiq as giq
from garmentiq.segmentation.model_definition.birefnet import (
    BiRefNet,
    load_birefnet_config,
)
from garmentiq.segmentation.model_definition.sam import (
    SamModel,
    Sam3Model,
    load_sam_config,
    load_sam_processor,
)
from garmentiq.matting.model_definition.vitmatte import (
    VitMatteForImageMatting,
    load_vitmatte_config,
    load_vitmatte_processor,
)
from garmentiq.matting.model_definition.mam import load_mam

# Matting runs at full image resolution. Apple Silicon ("mps") can return degenerate
# output on very large images, so CUDA or CPU is preferred here.
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

In [ ]:
# @title Download the test image and the model weights

!mkdir -p ./test_image
!wget -q -O ./test_image/cloth_1.jpg \
    https://raw.githubusercontent.com/lygitdata/GarmentIQ/refs/heads/gh-pages/asset/img/cloth_1.jpg

# BiRefNet, a prompt-free segmentation model
!mkdir -p ./models/birefnet
!wget -q -O ./models/birefnet/model.safetensors \
    https://huggingface.co/lygitdata/BiRefNet_garmentiq_backup/resolve/main/model.safetensors

# ViTMatte, small
!mkdir -p ./models/vitmatte
!wget -q -O ./models/vitmatte/model.safetensors \
    https://huggingface.co/hustvl/vitmatte-small-composition-1k/resolve/main/model.safetensors

# SAM 1, base
!mkdir -p ./models/sam_b
!wget -q -O ./models/sam_b/model.safetensors \
    https://huggingface.co/facebook/sam-vit-base/resolve/main/model.safetensors

# Matting Anything, about 408 MB. It bundles a frozen SAM, but GarmentIQ reads only the
# matting decoder from it and reuses the SAM you load separately.
!mkdir -p ./models/mam
!wget -q -O ./models/mam/mam_sam_vitb.pth \
    https://huggingface.co/spaces/shi-labs/Matting-Anything/resolve/main/checkpoints/mam_sam_vitb.pth

# SAM 3 is gated, see the SAM 3 section below.
!mkdir -p ./models/sam3

print("Downloads finished.")

<a name="vitmatte"></a>
## Matting with ViTMatte and a trimap

ViTMatte is guided by a **trimap**: white (255) is definitely foreground, black (0) is
definitely background, and gray (128) is the unknown band where the model is free to
predict soft alpha.

`generate_trimap` builds one from any segmentation mask. It erodes the mask to get
confident foreground, dilates it to get confident background, and marks the ring between
them as unknown. We start from BiRefNet, which needs no prompt.

In [ ]:
birefnet = giq.segmentation.load_model(
    model_class=BiRefNet,
    model_path="./models/birefnet/model.safetensors",
    model_args=load_birefnet_config(),
    device=device,
)

original_img, mask_biref = giq.segmentation.extract(
    model=birefnet,
    image_path="./test_image/cloth_1.jpg",
    resize_dim=(1024, 1024),
    normalize_mean=[0.485, 0.456, 0.406],
    normalize_std=[0.229, 0.224, 0.225],
    device=device,
)

trimap = giq.matting.generate_trimap(mask_biref, erode_size=15, dilate_size=15)

print("trimap values:", np.unique(trimap).tolist())
giq.segmentation.plot(image_np=trimap, figsize=(3, 3))

Now run ViTMatte. `composite` blends the image onto a new background
using the alpha values, rather than cutting it out along a hard edge.

In [ ]:
vitmatte = giq.matting.load_model(
    model_class=VitMatteForImageMatting,
    model_path="./models/vitmatte/model.safetensors",
    model_args={"config": load_vitmatte_config("vitmatte-small-composition-1k")},
    device=device,
)
vitmatte_processor = load_vitmatte_processor("vitmatte-small-composition-1k")

image_np, alpha = giq.matting.matte(
    model=vitmatte,
    image_path="./test_image/cloth_1.jpg",
    processor=vitmatte_processor,
    trimap=trimap,
    device=device,
)

composited = giq.matting.composite(
    image_np=image_np,
    alpha_np=alpha,
    background_color=(102, 255, 102),
)

giq.segmentation.plot(image_np=alpha, figsize=(3, 3))
giq.segmentation.plot(image_np=composited, figsize=(3, 3))

### Let GarmentIQ build the trimap

Passing `mask=` instead of `trimap=` derives the trimap internally, which is the usual way
to chain segmentation into matting. `trimap_args` forwards options to `generate_trimap`.

In [ ]:
_, alpha_auto = giq.matting.matte(
    model=vitmatte,
    image_path="./test_image/cloth_1.jpg",
    processor=vitmatte_processor,
    mask=mask_biref,
    trimap_args={"erode_size": 15, "dilate_size": 15},
    device=device,
)

print("identical to passing the trimap explicitly:", np.array_equal(alpha, alpha_auto))

### Tuning the unknown band

The band width decides where soft alpha may appear. Too narrow and the edges stay hard;
too wide and the model has to guess across large areas.

In [ ]:
for size in [5, 15, 35]:
    tm = giq.matting.generate_trimap(mask_biref, erode_size=size, dilate_size=size)
    _, a = giq.matting.matte(
        model=vitmatte,
        image_path="./test_image/cloth_1.jpg",
        processor=vitmatte_processor,
        trimap=tm,
        device=device,
    )
    soft = int(((a > 10) & (a < 245)).sum())
    print(f"erode/dilate={size:>2} | unknown band px={int((tm == 128).sum()):>9,} "
          f"| soft alpha px={soft:>9,}")

<a name="mam"></a>
## Matting with Matting Anything

Matting Anything needs no trimap. It prompts a frozen SAM and refines the resulting mask
into an alpha matte directly, so it is structurally tied to SAM and takes a SAM style
`prompt` instead of a trimap.

`load_mam` reads only the matting decoder from the checkpoint and pairs it with a SAM model
you already loaded, so SAM is not loaded twice.

In [ ]:
sam = giq.segmentation.load_model(
    model_class=SamModel,
    model_path="./models/sam_b/model.safetensors",
    model_args={"config": load_sam_config("sam-vit-b")},
    device=device,
)
sam_processor = load_sam_processor("sam-vit-b")

mam = load_mam(
    checkpoint_path="./models/mam/mam_sam_vitb.pth",
    sam_model=sam,
    sam_processor=sam_processor,
    device=device,
)

> **Prompt quality matters.** MAM keeps SAM's highest-confidence mask, and
> an ambiguous prompt can make SAM return a sub-part or even the background, which inverts
> the matte. A box drawn tightly around the garment is the reliable choice, so here we
> derive one from a segmentation pass first.

In [ ]:
_, mask_sam = giq.segmentation.extract(
    model=sam,
    image_path="./test_image/cloth_1.jpg",
    processor=sam_processor,
    prompt={"boxes": [[[200, 200, 1600, 2200]]]},
    device=device,
)

ys, xs = np.where(mask_sam > 127)
tight_box = [float(xs.min()), float(ys.min()), float(xs.max()), float(ys.max())]
print("tight garment box:", [round(v) for v in tight_box])

# Note there is no trimap anywhere in this call
image_np, alpha_mam = giq.matting.matte(
    model=mam,
    image_path="./test_image/cloth_1.jpg",
    prompt={"boxes": [[tight_box]]},
    device=device,
)

composited_mam = giq.matting.composite(
    image_np=image_np,
    alpha_np=alpha_mam,
    background_color=(102, 255, 102),
)

giq.segmentation.plot(image_np=alpha_mam, figsize=(3, 3))
giq.segmentation.plot(image_np=composited_mam, figsize=(3, 3))

<a name="sam3"></a>
## Matting from a text prompt with SAM 3

ViTMatte accepts a trimap from *any* segmentation mask, so it is not limited to BiRefNet.
Here the mask comes from SAM 3, which selects the garment from a natural-language
description.

> **SAM 3 is gated.** Accept the licence at
> [facebook/sam3](https://huggingface.co/facebook/sam3), download `model.safetensors`, and
> place it at `./models/sam3/model.safetensors`. Only the weights are gated: GarmentIQ
> bundles SAM 3's configuration, processor, and tokenizer.

In [ ]:
import os

SAM3_WEIGHTS = "./models/sam3/model.safetensors"
HAS_SAM3 = os.path.exists(SAM3_WEIGHTS)

if not HAS_SAM3:
    print(f"SAM 3 weights not found at {SAM3_WEIGHTS} - the SAM 3 cells will be skipped.")
else:
    print("SAM 3 weights found.")

In [ ]:
if HAS_SAM3:
    sam3 = giq.segmentation.load_model(
        model_class=Sam3Model,
        model_path=SAM3_WEIGHTS,
        model_args={"config": load_sam_config("sam3")},
        device=device,
    )

    _, mask_sam3 = giq.segmentation.extract(
        model=sam3,
        image_path="./test_image/cloth_1.jpg",
        processor=load_sam_processor("sam3"),
        prompt={"text": "t-shirt"},
        device=device,
    )

    # Exactly the same matting call as before, only the mask differs
    image_np, alpha_sam3 = giq.matting.matte(
        model=vitmatte,
        image_path="./test_image/cloth_1.jpg",
        processor=vitmatte_processor,
        mask=mask_sam3,
        trimap_args={"erode_size": 15, "dilate_size": 15},
        device=device,
    )

    composited_sam3 = giq.matting.composite(
        image_np=image_np,
        alpha_np=alpha_sam3,
        background_color=(102, 255, 102),
    )

    giq.segmentation.plot(image_np=alpha_sam3, figsize=(3, 3))
    giq.segmentation.plot(image_np=composited_sam3, figsize=(3, 3))

<a name="choosing"></a>
## Choosing a pairing

The two matting models differ in what they need to be guided by, which is what decides the
segmentation model they pair with.

| Model | Guidance | Trimap | Notes |
|---|---|:---:|---|
| **ViTMatte** | a trimap | required | Configs bundled for `small` and `base`; the trimap can be derived from any segmentation mask automatically |
| **Matting Anything** | a SAM prompt | not needed | Refines a SAM mask, and reuses the same SAM you already loaded for segmentation |

That gives three natural pairings:

| Segmentation | Matting | Guidance | Prompt needed |
|---|---|---|---|
| BiRefNet | ViTMatte | trimap from the mask | no, fully automatic |
| SAM 1 or SAM 2 | Matting Anything | a SAM prompt, points or boxes | yes, geometric |
| SAM 3 | ViTMatte | trimap from a text-prompted mask | yes, text |

ViTMatte works from any mask, so it pairs with any segmentation model. Matting Anything
consumes SAM's own image embeddings rather than just its mask, so it only works with SAM.

> **Apple Silicon note.** Above roughly 3.5 megapixels the MPS backend has been observed to
> return a degenerate ViTMatte matte that differs substantially from the CPU result, and it
> can exhaust GPU memory. GarmentIQ emits a `RuntimeWarning` in that case. Use
> `device="cpu"`, or downscale the image, for very large inputs.

To run matting as part of the full measurement pipeline, see the
[tailor tutorial](https://colab.research.google.com/github/lygitdata/GarmentIQ/blob/main/test/tutorial_tailor.ipynb).